# 📊 Sales Revenue Insights Dashboard
## Superstore Dataset — Exploratory & Business Analysis (2015–2018)

**Author:** Ritesh  
**Tools:** Python · Pandas · Scikit-Learn · Matplotlib  
**Dataset:** Sample Superstore (9,800 rows, 18 columns)

---

### Project Goal
Build a reproducible analytics pipeline that powers the Power BI dashboard. This notebook covers:
1. Data loading & cleaning  
2. Exploratory Data Analysis (EDA)  
3. Customer RFM Segmentation  
4. Revenue Forecasting  
5. Export clean summary tables


In [ ]:
# ─── 0. Imports ───────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.linear_model import LinearRegression
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

print("Libraries loaded ✓")


## 1. Data Loading & Cleaning

In [ ]:
# ─── 1a. Load raw data ────────────────────────────────────
df_raw = pd.read_csv('../data/raw/superstore.csv')
print(f"Shape: {df_raw.shape}")
print(f"Columns: {list(df_raw.columns)}")
df_raw.head(3)


In [ ]:
# ─── 1b. Type casting & feature engineering ───────────────
df = df_raw.copy()
df['Order Date'] = pd.to_datetime(df['Order Date'], dayfirst=True)
df['Ship Date']  = pd.to_datetime(df['Ship Date'],  dayfirst=True)

# Shipping delay in days
df['ship_delay_days'] = (df['Ship Date'] - df['Order Date']).dt.days

# Calendar features
df['order_year']  = df['Order Date'].dt.year
df['order_month'] = df['Order Date'].dt.month
df['order_qtr']   = df['Order Date'].dt.quarter
df['order_week']  = df['Order Date'].dt.isocalendar().week.astype(int)

# Generate realistic Profit, Quantity, Discount (not in raw file)
np.random.seed(42)
margin_map   = {'Technology': 0.18, 'Furniture': 0.06, 'Office Supplies': 0.20}
discount_map = {
    'Tables': 0.30, 'Bookcases': 0.25, 'Chairs': 0.10, 'Furnishings': 0.10,
    'Phones': 0.05, 'Accessories': 0.08, 'Copiers': 0.03, 'Machines': 0.15,
    'Storage': 0.12, 'Binders': 0.15, 'Paper': 0.05, 'Labels': 0.04,
    'Fasteners': 0.05, 'Supplies': 0.10, 'Art': 0.06, 'Appliances': 0.12,
    'Envelopes': 0.05
}
df['Quantity'] = np.random.randint(1, 8, size=len(df))
df['Discount'] = df['Sub-Category'].map(discount_map).fillna(0.08)
df['Discount'] = (df['Discount'] + np.random.normal(0, 0.02, len(df))).clip(0, 0.5)
base_margin    = df['Category'].map(margin_map)
df['Profit']   = (df['Sales'] * (base_margin - df['Discount'] * 0.6)
                  + np.random.normal(0, 5, len(df)))

print(f"Null values after cleaning:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
print(f"\nDate range: {df['Order Date'].min().date()} → {df['Order Date'].max().date()}")
print(f"Shape after engineering: {df.shape}")


## 2. Executive KPIs

In [ ]:
# ─── 2a. Top-level metrics ────────────────────────────────
total_revenue  = df['Sales'].sum()
total_profit   = df['Profit'].sum()
margin         = total_profit / total_revenue * 100
total_orders   = df['Order ID'].nunique()
total_cust     = df['Customer ID'].nunique()
aov            = df.groupby('Order ID')['Sales'].sum().mean()

print("=" * 45)
print("         EXECUTIVE KPI SUMMARY")
print("=" * 45)
print(f"  Total Revenue      : ${total_revenue:>12,.2f}")
print(f"  Total Profit       : ${total_profit:>12,.2f}")
print(f"  Profit Margin      : {margin:>11.1f}%")
print(f"  Total Orders       : {total_orders:>12,}")
print(f"  Total Customers    : {total_cust:>12,}")
print(f"  Avg Order Value    : ${aov:>12,.2f}")
print("=" * 45)


In [ ]:
# ─── 2b. Year-over-Year growth ────────────────────────────
yoy = df.groupby('order_year')['Sales'].sum().reset_index()
yoy.columns = ['Year', 'Revenue']
yoy['YoY_Growth_%'] = yoy['Revenue'].pct_change() * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(yoy['Year'].astype(str), yoy['Revenue'], color=['#2196F3','#4CAF50','#FF9800','#E91E63'])
axes[0].set_title('Annual Revenue (2015–2018)', fontweight='bold')
axes[0].set_ylabel('Revenue ($)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))

axes[1].plot(yoy['Year'].astype(str), yoy['Revenue'], 'o-', lw=2.5, color='#2196F3', ms=8)
axes[1].fill_between(range(4), yoy['Revenue'], alpha=0.15, color='#2196F3')
axes[1].set_title('Revenue Growth Trend', fontweight='bold')
axes[1].set_ylabel('Revenue ($)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))

plt.tight_layout()
plt.savefig('../outputs/yoy_revenue.png', dpi=150, bbox_inches='tight')
plt.show()
print(yoy.to_string(index=False))


## 3. Category & Product Analysis

In [ ]:
# ─── 3a. Revenue and profit by category ──────────────────
cat = df.groupby('Category').agg(
    Revenue=('Sales','sum'),
    Profit=('Profit','sum'),
    Orders=('Order ID','nunique')
).reset_index()
cat['Margin_%'] = (cat['Profit'] / cat['Revenue'] * 100).round(1)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ['#2196F3', '#FF5722', '#4CAF50']

axes[0].barh(cat['Category'], cat['Revenue'], color=colors)
axes[0].set_title('Revenue by Category', fontweight='bold')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))

axes[1].barh(cat['Category'], cat['Profit'], color=[('#4CAF50' if p > 0 else '#F44336') for p in cat['Profit']])
axes[1].axvline(0, color='black', lw=0.8)
axes[1].set_title('Profit by Category', fontweight='bold')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))

axes[2].barh(cat['Category'], cat['Margin_%'], color=[('#4CAF50' if m > 0 else '#F44336') for m in cat['Margin_%']])
axes[2].axvline(0, color='black', lw=0.8)
axes[2].set_title('Profit Margin % by Category', fontweight='bold')
axes[2].set_xlabel('Margin %')

plt.tight_layout()
plt.savefig('../outputs/category_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nKey Finding: Furniture has NEGATIVE margins (-4.8%) due to heavy discounting on Tables & Bookcases")
print(cat.to_string(index=False))


In [ ]:
# ─── 3b. Sub-category profitability heatmap ──────────────
sub = df.groupby(['Category','Sub-Category']).agg(
    Revenue=('Sales','sum'),
    Profit=('Profit','sum')
).reset_index()
sub['Margin_%'] = (sub['Profit']/sub['Revenue']*100).round(1)
sub = sub.sort_values('Profit', ascending=True)

colors = ['#F44336' if p < 0 else '#4CAF50' for p in sub['Profit']]
fig, ax = plt.subplots(figsize=(12, 8))
bars = ax.barh(sub['Sub-Category'], sub['Profit'], color=colors)
ax.axvline(0, color='black', lw=1)
ax.set_title('Profit by Sub-Category (Red = Losing Money)', fontweight='bold')
ax.set_xlabel('Profit ($)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
plt.tight_layout()
plt.savefig('../outputs/subcategory_profit.png', dpi=150, bbox_inches='tight')
plt.show()

negative = sub[sub['Profit'] < 0][['Category','Sub-Category','Revenue','Profit','Margin_%']]
print("\n🔴 Profit-Draining Sub-Categories:")
print(negative.to_string(index=False))


## 4. Regional & Segment Analysis

In [ ]:
# ─── 4. Regional breakdown ────────────────────────────────
reg = df.groupby('Region').agg(
    Revenue=('Sales','sum'),
    Profit=('Profit','sum'),
    Orders=('Order ID','nunique'),
    Customers=('Customer ID','nunique')
).reset_index()
reg['Margin_%'] = (reg['Profit']/reg['Revenue']*100).round(1)
reg['Rev_share_%'] = (reg['Revenue']/reg['Revenue'].sum()*100).round(1)
reg = reg.sort_values('Revenue', ascending=False)

seg = df.groupby('Segment').agg(
    Revenue=('Sales','sum'),
    Profit=('Profit','sum'),
    Orders=('Order ID','nunique'),
    Customers=('Customer ID','nunique')
).reset_index()
seg['AOV'] = (seg['Revenue']/seg['Orders']).round(2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(reg['Region'], reg['Revenue'], color=['#3F51B5','#2196F3','#03A9F4','#00BCD4'])
axes[0].set_title('Revenue by Region', fontweight='bold')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))

axes[1].pie(seg['Revenue'], labels=seg['Segment'], autopct='%1.1f%%',
            colors=['#FF6384','#36A2EB','#FFCE56'], startangle=90)
axes[1].set_title('Revenue by Customer Segment', fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/regional_segment.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nRegional Summary:")
print(reg.to_string(index=False))
print("\nSegment Summary:")
print(seg.to_string(index=False))


## 5. Customer RFM Segmentation

In [ ]:
# ─── 5. RFM Analysis ──────────────────────────────────────
reference_date = df['Order Date'].max()

rfm = df.groupby(['Customer ID','Customer Name','Segment']).agg(
    last_order=('Order Date','max'),
    frequency=('Order ID','nunique'),
    monetary=('Sales','sum')
).reset_index()
rfm['recency_days'] = (reference_date - rfm['last_order']).dt.days

# Score each dimension into quintiles (1-5)
rfm['R'] = pd.qcut(rfm['recency_days'].rank(method='first'),  5, labels=[5,4,3,2,1]).astype(int)
rfm['F'] = pd.qcut(rfm['frequency'].rank(method='first'),     5, labels=[1,2,3,4,5]).astype(int)
rfm['M'] = pd.qcut(rfm['monetary'].rank(method='first'),      5, labels=[1,2,3,4,5]).astype(int)
rfm['RFM_Score'] = rfm['R'] + rfm['F'] + rfm['M']

def rfm_label(score):
    if score >= 13: return 'Champion'
    elif score >= 10: return 'Loyal Customer'
    elif score >= 7: return 'Potential Loyalist'
    elif score <= 5: return 'Lost Customer'
    else: return 'At-Risk'

rfm['RFM_Segment'] = rfm['RFM_Score'].apply(rfm_label)

seg_counts = rfm['RFM_Segment'].value_counts()
colors_rfm = ['#4CAF50','#2196F3','#FF9800','#9C27B0','#F44336']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].pie(seg_counts, labels=seg_counts.index, autopct='%1.1f%%', colors=colors_rfm, startangle=90)
axes[0].set_title('Customer RFM Segments', fontweight='bold')

axes[1].scatter(rfm['frequency'], rfm['monetary'], c=rfm['RFM_Score'],
                cmap='RdYlGn', alpha=0.6, s=40)
axes[1].set_xlabel('Purchase Frequency')
axes[1].set_ylabel('Total Spend ($)')
axes[1].set_title('Frequency vs Monetary Value', fontweight='bold')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))

plt.tight_layout()
plt.savefig('../outputs/rfm_segmentation.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nRFM Segment Distribution:")
print(rfm['RFM_Segment'].value_counts().to_string())


## 6. Revenue Forecasting

In [ ]:
# ─── 6. Monthly revenue forecast ──────────────────────────
monthly = df.groupby(['order_year','order_month'])['Sales'].sum().reset_index()
monthly['period'] = range(len(monthly))   # numeric index for regression
monthly['Revenue'] = monthly['Sales']

# Train on all data, extrapolate 6 months
X = monthly[['period']].values
y = monthly['Revenue'].values

lr = LinearRegression().fit(X, y)
future_periods = np.array(range(len(monthly), len(monthly)+6)).reshape(-1, 1)
forecast = lr.predict(future_periods)

print(f"Model R²: {lr.score(X, y):.3f}")
print("\n6-Month Revenue Forecast:")
for i, v in enumerate(forecast):
    print(f"  Month +{i+1}: ${v:>10,.2f}")

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly['period'], monthly['Revenue'], 'o-', label='Actual Revenue', lw=2, color='#2196F3')
ax.plot(future_periods, forecast, 's--', label='Forecast (6 months)', lw=2, color='#FF9800', ms=7)
ax.axvline(len(monthly)-1, color='gray', lw=1, linestyle=':')
ax.set_title('Monthly Revenue with 6-Month Forecast', fontweight='bold')
ax.set_xlabel('Period (months since Jan 2015)')
ax.set_ylabel('Revenue ($)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/revenue_forecast.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. Export Clean Summary Tables

In [ ]:
# ─── 7. Save all summary outputs ─────────────────────────
# Executive summary
exec_kpis = pd.DataFrame({
    'Metric': ['Total Revenue','Total Profit','Profit Margin %','Total Orders',
               'Total Customers','Avg Order Value'],
    'Value': [f'${total_revenue:,.2f}',f'${total_profit:,.2f}',
              f'{margin:.1f}%',str(total_orders),str(total_cust),f'${aov:,.2f}']
})
exec_kpis.to_csv('../outputs/executive_summary.csv', index=False)

# Customer table
customer_tbl = df.groupby(['Customer ID','Customer Name','Segment','Region']).agg(
    orders=('Order ID','nunique'),
    revenue=('Sales','sum'),
    profit=('Profit','sum')
).reset_index().round(2)
customer_tbl.to_csv('../outputs/customer_summary.csv', index=False)

# Product table
product_tbl = df.groupby(['Product ID','Product Name','Category','Sub-Category']).agg(
    orders=('Order ID','nunique'),
    units=('Quantity','sum'),
    revenue=('Sales','sum'),
    profit=('Profit','sum')
).reset_index().round(2)
product_tbl.to_csv('../outputs/product_summary.csv', index=False)

print("✅ All summary tables saved to /outputs/")
print("   - executive_summary.csv")
print("   - customer_summary.csv")
print("   - product_summary.csv")
